### 1. Import Libraries & Load Dataset
- **Objective:** Import all necessary Python libraries for data manipulation, visualization, regression modeling, and evaluation.
- **Dataset:** Loaded the built-in **California Housing Dataset** from `sklearn.datasets`.
- **Details:** Displayed the first few rows and the shape of the dataset to understand the features (like Median Income, House Age, Ave Rooms, etc.) and the target variable (`MedHouseValue`).


In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load California Housing Dataset
housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['MedHouseValue'] = housing.target

print(df.head())
print("\nDataset Shape:", df.shape)

   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  MedHouseValue  
0    -122.23          4.526  
1    -122.22          3.585  
2    -122.24          3.521  
3    -122.25          3.413  
4    -122.25          3.422  

Dataset Shape: (20640, 9)


### **Result & Analysis:**
- The **California Housing Dataset** has been successfully imported with a total of **20,640 samples and 9 features** (including the target variable `MedHouseValue`).
- Features like `MedHouseAge`, `AveRooms`, `Population`, and `Latitude/Longitude` are loaded properly without any initial loading errors, providing a solid foundation for regression modeling.


### 2. Data Preprocessing & Missing Value Check
- **Missing Values:** Checked for any null or missing values to ensure data cleanliness.
- **Feature Selection:** Separated the dataset into features (`X`) and the target variable (`y`).
- **Train-Test Split:** Split the data into training (80%) and testing (20%) sets using `train_test_split` with a fixed `random_state` for reproducibility.
- **Feature Scaling:** Applied `StandardScaler` to scale the features, ensuring algorithms like SVR and Linear Regression perform optimally.


In [2]:

# Check for missing values
print("Missing values:\n", df.isnull().sum())

# Define features (X) and target (y)
X = df.drop('MedHouseValue', axis=1)
y = df['MedHouseValue']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature Scaling using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Missing values:
 MedInc           0
HouseAge         0
AveRooms         0
AveBedrms        0
Population       0
AveOccup         0
Latitude         0
Longitude        0
MedHouseValue    0
dtype: int64


### **Result & Analysis:**
- **Missing Value Check:** The output confirms **0 missing values** across all columns, meaning no imputation is required, ensuring data integrity.
- **Train-Test Split:** The dataset is successfully split into an 80-20 partition, providing a robust training set to learn patterns and an independent test set to evaluate unseen performance.
- **Feature Scaling:** `StandardScaler` successfully standardized the features, which is essential for distance-sensitive algorithms like SVR and Linear Regression to converge efficiently.


### 3. Model Implementation & Evaluation
- **Models Implemented:** 
  1. Linear Regression
  2. Decision Tree Regressor
  3. Random Forest Regressor
  4. Gradient Boosting Regressor
  5. Support Vector Regressor (SVR)
- **Evaluation Metrics:** Evaluated all models using **Mean Squared Error (MSE)**, **Mean Absolute Error (MAE)**, and **R-squared Score ($R^2$)**.
- **Result Analysis:** Compiled the evaluation metrics into a pandas DataFrame to easily compare and identify the best-performing algorithm.


In [3]:

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "Support Vector Regressor": SVR()
}

results = []

for name, model in models.items():
    # Use scaled data for SVR and Linear Regression, raw/scaled works for trees
    if name in ["Linear Regression", "Support Vector Regressor"]:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results.append({"Model": name, "MSE": mse, "MAE": mae, "R2 Score": r2})

results_df = pd.DataFrame(results)
print(results_df)

                      Model       MSE       MAE  R2 Score
0         Linear Regression  0.555892  0.533200  0.575788
1             Decision Tree  0.495235  0.454679  0.622076
2             Random Forest  0.255368  0.327543  0.805123
3         Gradient Boosting  0.293997  0.371643  0.775645
4  Support Vector Regressor  0.357004  0.398599  0.727563


### **Result & Analysis:**
- **Performance Comparison:** 
  - **Random Forest Regressor** achieved the highest performance with the lowest MSE/MAE and the highest **$R^2$ Score (~0.805)**, meaning it explains about 80.5% of the variance in housing prices.
  - **Gradient Boosting** followed closely as the second-best model (R2 Score ~0.775).
  - **Linear Regression** performed the lowest among the tested models, indicating that the relationships in the housing data are non-linear and better captured by ensemble tree-based models.
- **Conclusion:** Ensemble models (Random Forest and Gradient Boosting) significantly outperform linear and support vector approaches on this dataset.


### 4. Cross-Validation & Hyperparameter Tuning
- **Cross-Validation:** Performed 5-fold cross-validation (`cv=5`) to ensure the model's performance is robust and not dependent on a specific train-test split.
- **Hyperparameter Tuning:** Used `RandomizedSearchCV` to fine-tune hyperparameters (like `n_estimators`, `max_depth`, and `min_samples_split`) for the Random Forest model to optimize its predictive accuracy.


In [4]:

from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5]
}

rf = RandomForestRegressor(random_state=42)
rf_cv = RandomizedSearchCV(estimator=rf, param_distributions=param_grid, n_iter=3, cv=5, scoring='r2', random_state=42, n_jobs=-1)
rf_cv.fit(X_train, y_train)

print("Tuned Best Parameters:", rf_cv.best_params_)
print("Best Cross-Validation R2 Score:", rf_cv.best_score_)

Tuned Best Parameters: {'n_estimators': 200, 'min_samples_split': 2, 'max_depth': 20}
Best Cross-Validation R2 Score: 0.8050524232098117


### **Result & Analysis:**
- **Robustness:** 5-fold cross-validation confirmed that the high performance of the Random Forest model is consistent and not due to a lucky train-test split.
- **Hyperparameter Tuning:** Using `RandomizedSearchCV` helped optimize the hyperparameters (such as `n_estimators`, `max_depth`), yielding a strong and stable cross-validation score, ensuring the model generalizes well to new data.
